In [2]:
"""
innovation_final.py
===================
FINAL INNOVATION MODULE — RL Music Emotion Management
Based on: Dutta et al., IEEE EMBC 2020

Merges:
  • CSV-only data loading with code-1 normalisation pipeline (z-score → tanh)
  • Action space = 8 GEW emotion directions (inspired by code 2)
  • Clip selection via best_clip(clips, action_idx, vs) — cosine similarity
  • All 4 innovations from original innovation.py (DQN, Momentum, Adaptive Shaping, Multi-Target)

Innovations:
┌──────────────────────────────────────────────────────────────────────┐
│ INN-1  Continuous Deep Q-Network                                     │
│        • Neural network replaces 9-bucket Q-table                   │
│        • State = raw (valence, arousal) ∈ [-1,1]² — no rounding     │
│        • Generalises across similar emotional states                 │
│                                                                      │
│ INN-2  Momentum-Augmented State                                      │
│        • State = (v, a, Δv, Δa)  4-D instead of 2-D               │
│        • Agent perceives emotional velocity → avoids oscillation     │
│                                                                      │
│ INN-3  Adaptive Reward Shaping                                       │
│        • λ(t) = λ_base × (1 + β · t/T) grows with iteration depth  │
│        • Free exploration early; heavy stagnation-penalty later      │
│                                                                      │
│ INN-4  Multi-Target Emotion Navigation                               │
│        • Trains/evaluates for EVERY GEW emotion as target           │
│        • Maps which emotional destinations are easy vs hard to reach │
└──────────────────────────────────────────────────────────────────────┘
"""

import os, warnings, random, pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from collections import deque

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(it, desc="", **kw):
        it = list(it); print(f"  {desc}: {len(it)} items"); return it

warnings.filterwarnings("ignore")
np.random.seed(42)
random.seed(42)

# ══════════════════════════════════════════════════════════════════════════════
# 0.  OUTPUT DIRECTORY
# ══════════════════════════════════════════════════════════════════════════════
OUT_DIR = "./outputs_innovation"
os.makedirs(OUT_DIR, exist_ok=True)
def out(fname): return os.path.join(OUT_DIR, fname)

# ══════════════════════════════════════════════════════════════════════════════
# 1.  PAPER CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
ALPHA           = 0.1
GAMMA           = 0.6
EPSILON         = 0.1
LAMBDA_BASE     = -100.0        # fixed shaping penalty (paper)
N_ITERATIONS    = 6             # clips per playlist
N_SUBJECTS      = 32
N_CLIPS         = 40
TRAIN_EPOCHS    = 1000          # tabular Q epochs (matches baseline)
TARGET_EMOTION  = "happy"
CONVERGE_THRESH = 10.0          # degrees — matches baseline paper threshold

# DQN hyper-parameters (INN-1 / INN-2)
DQN_LR         = 5e-3
DQN_HIDDEN     = 48
DQN_BUFFER_CAP = 1500
DQN_BATCH      = 16
DQN_TARGET_UPD = 40
DQN_EPOCHS     = 80

# Adaptive shaping (INN-3)
ADAPT_BETA     = 0.8            # λ(t) = λ_base*(1 + β*t/T)

# Multi-target (INN-4)
MT_EPOCHS      = 60

# ══════════════════════════════════════════════════════════════════════════════
# 2.  GEW — 8 primary emotions + NEUTRAL start state
#     (coordinates from baseline code — paper-faithful)
# ══════════════════════════════════════════════════════════════════════════════
GEW_EMOTIONS = {
    "sadness":  [-0.81, -0.71],
    "pride":    [ 0.73,  0.04],
    "interest": [-0.22,  0.71],
    "happy":    [ 0.73,  0.75],
    "guilt":    [-0.21, -0.81],
    "disgust":  [-0.37, -0.06],
    "anger":    [-0.85,  0.69],
    "relief":   [ 0.59, -0.67],
}
NEUTRAL_STATE  = "neutral"
EMOTION_NAMES  = list(GEW_EMOTIONS.keys())
ALL_STATES     = [NEUTRAL_STATE] + EMOTION_NAMES
STATE_INDEX    = {s: i for i, s in enumerate(ALL_STATES)}
N_STATES       = len(ALL_STATES)     # 9
N_ACTIONS      = len(EMOTION_NAMES)  # 8  ← action space = 8 GEW emotion directions

# ── unit helpers ───────────────────────────────────────────────
def _unit(v):
    v = np.asarray(v, dtype=float)
    n = np.linalg.norm(v)
    return v / n if n > 1e-9 else v

def _is_neutral_vec(v):
    return np.linalg.norm(np.asarray(v, float)) < 1e-9

GEW_UNIT      = {k: _unit(v) for k, v in GEW_EMOTIONS.items()}
TARGET_VEC    = GEW_UNIT[TARGET_EMOTION]
ACTION_VEC    = np.array([GEW_UNIT[e] for e in EMOTION_NAMES])  # (8,2)
# action index ↔ emotion name (mirrors code-2 convention)
ACTION_INDEX  = {name: i for i, name in enumerate(EMOTION_NAMES)}

# Evaluation start: anger (matches baseline EVAL_START_EMOTION)
EVAL_FROM_NEUTRAL  = False
EVAL_START_EMOTION = "anger"

EMOTION_COLORS = {
    "neutral":  "#AAAAAA", "happy":    "#FFD700", "pride":    "#90EE90",
    "interest": "#00CED1", "anger":    "#FF4500", "disgust":  "#FF8C00",
    "guilt":    "#9370DB", "sadness":  "#4169E1", "relief":   "#FF69B4",
}

# ══════════════════════════════════════════════════════════════════════════════
# 3.  DATA LOADING  — CSV only, code-1 normalisation pipeline (z-score → tanh)
# ══════════════════════════════════════════════════════════════════════════════
def _normalise_clips(raw):
    """
    raw : (n_clips, 2) float array  [valence, arousal]
    Pipeline: z-score per-subject → tanh squash  (code-1 / paper-faithful)
    """
    mean = raw.mean(axis=0)
    std  = raw.std(axis=0, ddof=1)
    std  = np.where(std < 1e-6, 1.0, std)
    z    = (raw - mean) / std
    return np.tanh(z).astype(float)


def load_deap(csv_path="./deap-dataset/Metadata/participant_ratings.csv"):
    """
    Load DEAP from CSV metadata file.
    Sorts by Participant_id then Experiment_id for correct clip ordering,
    then applies code-1 normalisation: z-score per subject → tanh squash.
    Returns dict {subject_id: (N_clips, 2) float array}.
    """
    if not os.path.exists(csv_path):
        print(f"  [Error] CSV not found: {csv_path}")
        return None
    df = pd.read_csv(csv_path)
    df = df.sort_values(by=["Participant_id", "Experiment_id"]).reset_index(drop=True)
    subjects = {}
    for sid, group in df.groupby("Participant_id"):
        raw = group[["Valence", "Arousal"]].values.astype(float)
        subjects[int(sid)] = _normalise_clips(raw)
    print(f"  Loaded DEAP (CSV): {len(subjects)} subjects, "
          f"{next(iter(subjects.values())).shape[0]} clips each.")
    return subjects


def make_synthetic(n_subjects=32, n_clips=40, seed=0):
    """Fallback synthetic data drawn from GEW cluster centres."""
    rng = np.random.default_rng(seed)
    subjects = {}
    for sid in range(1, n_subjects + 1):
        w      = rng.dirichlet(np.ones(N_ACTIONS) * 0.5)
        chosen = rng.choice(N_ACTIONS, size=n_clips, p=w)
        noise  = rng.normal(0, 0.25, (n_clips, 2))
        raw    = ACTION_VEC[chosen] + noise
        subjects[sid] = _normalise_clips(raw)
    print(f"  Synthetic DEAP: {n_subjects} subjects, {n_clips} clips each.")
    return subjects

# ══════════════════════════════════════════════════════════════════════════════
# 4.  STATE QUANTISATION
# ══════════════════════════════════════════════════════════════════════════════
def vec_to_state(vec):
    vec = np.asarray(vec, dtype=float)
    if _is_neutral_vec(vec):
        return NEUTRAL_STATE
    uvec = _unit(vec)
    best, best_cos = NEUTRAL_STATE, -np.inf
    for name, gvec in GEW_UNIT.items():
        c = float(np.dot(uvec, gvec))
        if c > best_cos:
            best_cos, best = c, name
    return best

# ══════════════════════════════════════════════════════════════════════════════
# 5.  CLIP SELECTION  (inspired by code 2)
#
#     The agent chooses one of 8 GEW emotion direction indices as its action.
#     best_clip() then finds the subject-specific clip whose tanh-normalised
#     vector best matches that emotion direction, blended with the current
#     state direction for smoother trajectories.
#
#     Key improvement over original code-1:
#       • Action space is subject-independent (always 8 GEW directions)
#       • Q-values / DQN weights transfer cleanly across subjects in LOSO
# ══════════════════════════════════════════════════════════════════════════════
def best_clip(clips, action_idx, vs):
    """
    clips      : (N, 2) tanh-normalised clip vectors for one subject
    action_idx : int, index into EMOTION_NAMES / ACTION_VEC
    vs         : (2,) current state vector

    Returns the clip index (int) whose direction best aligns with the chosen
    GEW emotion, blended 70/30 with the current state direction.
    """
    target_dir  = ACTION_VEC[action_idx]           # (2,) unit vector
    sims_target = clips @ target_dir               # cosine sim to chosen emotion

    if not _is_neutral_vec(vs):
        sims_state = clips @ _unit(vs)             # cosine sim to current state
    else:
        sims_state = np.zeros(len(clips))

    scores = 0.7 * sims_target + 0.3 * sims_state
    return int(np.argmax(scores))

# ══════════════════════════════════════════════════════════════════════════════
# 6.  TRANSITION  τ(s, a)
# ══════════════════════════════════════════════════════════════════════════════
def transition(vs, va):
    vs = np.asarray(vs, float)
    va = np.asarray(va, float)
    if _is_neutral_vec(vs):
        return np.clip(va.copy(), -1.0, 1.0)
    return np.clip(vs + va, -1.0, 1.0)

# ══════════════════════════════════════════════════════════════════════════════
# 7.  ANGULAR ERROR
# ══════════════════════════════════════════════════════════════════════════════
def _safe_angle_deg(v1, v2):
    u1 = _unit(np.asarray(v1, float))
    u2 = _unit(np.asarray(v2, float))
    if _is_neutral_vec(u1) or _is_neutral_vec(u2):
        return 180.0
    dot = np.clip(np.dot(u1, u2), -1.0, 1.0)
    return np.degrees(np.arccos(dot))

def angular_error(vs, target_vec=None):
    if target_vec is None: target_vec = TARGET_VEC
    return _safe_angle_deg(np.asarray(vs, float), target_vec)

# ══════════════════════════════════════════════════════════════════════════════
# 8.  REWARD  r(s, a)
# ══════════════════════════════════════════════════════════════════════════════
def reward_fn(vs, va, target_vec=None):
    if target_vec is None: target_vec = TARGET_VEC
    vs, va = np.asarray(vs, float), np.asarray(va, float)
    theta1 = _safe_angle_deg(target_vec, va)
    if _is_neutral_vec(vs):
        return 1.0 if theta1 <= 90.0 else 0.0
    theta2 = _safe_angle_deg(vs, va)
    return 1.0 if (theta1 + theta2) <= 180.0 else 0.0

# ══════════════════════════════════════════════════════════════════════════════
# 9.  SHAPING FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════
def _shaping_fixed(s, sp, **kwargs):
    """Paper's constant penalty for stagnation."""
    if s == sp and s != TARGET_EMOTION:
        return LAMBDA_BASE
    return 0.0

def _shaping_adaptive(s, sp, step=0, **kwargs):
    """INN-3: iteration-aware penalty that grows with step depth."""
    if s == sp and s != TARGET_EMOTION:
        t   = step / N_ITERATIONS
        lam = LAMBDA_BASE * (1.0 + ADAPT_BETA * t)
        return float(lam)
    return 0.0

# ══════════════════════════════════════════════════════════════════════════════
# 10.  START VECTOR HELPER
# ══════════════════════════════════════════════════════════════════════════════
def _make_start_vec(rng=None):
    if EVAL_FROM_NEUTRAL:
        return np.array([0.0, 0.0])
    base = GEW_UNIT[EVAL_START_EMOTION].copy()
    if rng is not None:
        base = _unit(base + rng.normal(0, 0.05, 2))
    return base

# ══════════════════════════════════════════════════════════════════════════════
# 11.  BASELINE — Tabular Q-learning (paper method, 9-state × 8-action)
#
#      Action space = 8 GEW emotion directions (code-2 inspired, subject-independent).
#      Agent picks emotion action index → best_clip() resolves the actual clip.
# ══════════════════════════════════════════════════════════════════════════════
class TabularQAgent:
    def __init__(self):
        # Q-table: 9 states × 8 GEW emotion direction actions
        self.Q = {s: np.random.uniform(-0.01, 0.01, N_ACTIONS) for s in ALL_STATES}

    def act(self, state, explore=True):
        if explore and np.random.rand() < EPSILON:
            return np.random.randint(N_ACTIONS)
        return int(np.argmax(self.Q[state]))

    def update(self, s, a, r, phi, sp):
        best_next = float(np.max(self.Q[sp]))
        tgt = r + GAMMA * best_next + phi
        self.Q[s][a] = (1 - ALPHA) * self.Q[s][a] + ALPHA * tgt


def run_loso_tabular(subjects, shaping_fn=None, verbose=True):
    """
    Standard LOSO with tabular Q and pluggable shaping function.
    Action = GEW emotion direction index (8, subject-independent).
    Actual clip resolved per-subject via best_clip().
    """
    if shaping_fn is None: shaping_fn = _shaping_fixed
    sids = sorted(subjects.keys())
    errors, state_paths, vec_paths = {}, {}, {}
    rng  = np.random.default_rng(0)

    iterator = tqdm(sids, desc="Tabular LOSO", disable=not verbose)
    for test_sid in iterator:
        train_sids = [s for s in sids if s != test_sid]
        agent = TabularQAgent()

        for epoch in range(TRAIN_EPOCHS):
            for tr_sid in train_sids:
                clips = subjects[tr_sid]          # (N_CLIPS, 2)
                vs = _make_start_vec(rng)
                s  = vec_to_state(vs)

                for step in range(N_ITERATIONS):
                    # 1. Agent picks GEW emotion action index (subject-independent)
                    a   = agent.act(s, explore=True)
                    # 2. Resolve to best matching clip for that emotion direction
                    va  = clips[best_clip(clips, a, vs)]
                    # 3. Transition & reward
                    vsp = transition(vs, va)
                    sp  = vec_to_state(vsp)
                    r   = reward_fn(vs, va)
                    phi = shaping_fn(s, sp, step=step)
                    agent.update(s, a, r, phi, sp)
                    vs, s = vsp, sp
                    if s == TARGET_EMOTION: break

        # ── Greedy evaluation ────────────────────────────────
        clips = subjects[test_sid]
        vs    = _make_start_vec()
        s     = vec_to_state(vs)
        errs, states, vecs = [], [s], [vs.copy()]

        for _ in range(N_ITERATIONS):
            a  = agent.act(s, explore=False)
            va = clips[best_clip(clips, a, vs)]
            vs = transition(vs, va)
            s  = vec_to_state(vs)
            errs.append(angular_error(vs))
            states.append(s)
            vecs.append(vs.copy())

        errors[test_sid]      = np.array(errs)
        state_paths[test_sid] = states
        vec_paths[test_sid]   = vecs

    return errors, state_paths, vec_paths

# ══════════════════════════════════════════════════════════════════════════════
# 12.  INN-1 & INN-2: CONTINUOUS DEEP Q-NETWORK (pure NumPy)
#
#      Action space = 8 GEW emotion directions (same as tabular agent).
#      DQN outputs Q-values for each of the 8 emotion direction actions.
#      Clip resolved per-subject via best_clip() after action selection.
# ══════════════════════════════════════════════════════════════════════════════
class _MLP:
    """3-layer fully-connected network: ReLU hidden, linear output."""
    def __init__(self, d_in, d_hidden, d_out, lr=DQN_LR):
        self.lr = lr
        k1 = np.sqrt(2.0 / d_in);      k2 = np.sqrt(2.0 / d_hidden)
        self.W1 = np.random.randn(d_in,     d_hidden) * k1
        self.b1 = np.zeros(d_hidden)
        self.W2 = np.random.randn(d_hidden, d_hidden) * k2
        self.b2 = np.zeros(d_hidden)
        self.W3 = np.random.randn(d_hidden, d_out)    * k2
        self.b3 = np.zeros(d_out)

    def predict(self, X):
        single = (X.ndim == 1)
        if single: X = X[np.newaxis]
        h1  = np.maximum(0, X  @ self.W1 + self.b1)
        h2  = np.maximum(0, h1 @ self.W2 + self.b2)
        out = h2 @ self.W3 + self.b3
        return out[0] if single else out

    def train_batch(self, X, target_Q):
        B = len(X)
        Z1 = X  @ self.W1 + self.b1
        A1 = np.maximum(0, Z1)
        Z2 = A1 @ self.W2 + self.b2
        A2 = np.maximum(0, Z2)
        out = A2 @ self.W3 + self.b3

        diff = out - target_Q
        dout = 2 * diff / B

        dW3 = A2.T @ dout;  db3 = dout.sum(0)
        dA2 = dout @ self.W3.T
        dZ2 = dA2 * (Z2 > 0)
        dW2 = A1.T @ dZ2;   db2 = dZ2.sum(0)
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * (Z1 > 0)
        dW1 = X.T  @ dZ1;   db1 = dZ1.sum(0)

        for g in [dW1, db1, dW2, db2, dW3, db3]:
            np.clip(g, -1.0, 1.0, out=g)

        self.W1 -= self.lr * dW1; self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2; self.b2 -= self.lr * db2
        self.W3 -= self.lr * dW3; self.b3 -= self.lr * db3
        return float(np.mean(diff ** 2))

    def copy_weights_from(self, other):
        self.W1 = other.W1.copy(); self.b1 = other.b1.copy()
        self.W2 = other.W2.copy(); self.b2 = other.b2.copy()
        self.W3 = other.W3.copy(); self.b3 = other.b3.copy()


class DQNAgent:
    """
    INN-1 (state_dim=2) / INN-2 (state_dim=4) DQN agent.
    Outputs Q-values over 8 GEW emotion direction actions.
    Uses experience replay and a target network for stability.
    """
    def __init__(self, state_dim=2, n_actions=N_ACTIONS):
        self.n_actions = n_actions
        self.q_net  = _MLP(state_dim, DQN_HIDDEN, n_actions)
        self.t_net  = _MLP(state_dim, DQN_HIDDEN, n_actions)
        self.t_net.copy_weights_from(self.q_net)
        self.buffer = deque(maxlen=DQN_BUFFER_CAP)
        self.steps  = 0

    def act(self, state_vec, explore=True):
        if explore and np.random.rand() < EPSILON:
            return np.random.randint(self.n_actions)
        return int(np.argmax(self.q_net.predict(np.asarray(state_vec, float))))

    def store(self, s, a, r, sp):
        self.buffer.append((np.asarray(s, float), int(a),
                            float(r), np.asarray(sp, float)))

    def learn(self):
        if len(self.buffer) < DQN_BATCH:
            return
        batch = random.sample(self.buffer, DQN_BATCH)
        S  = np.array([b[0] for b in batch])
        A  = np.array([b[1] for b in batch])
        R  = np.array([b[2] for b in batch])
        Sp = np.array([b[3] for b in batch])

        q_cur  = self.q_net.predict(S)
        q_next = self.t_net.predict(Sp).max(1)

        target_Q = q_cur.copy()
        target_Q[np.arange(DQN_BATCH), A] = R + GAMMA * q_next

        self.q_net.train_batch(S, target_Q)
        self.steps += 1
        if self.steps % DQN_TARGET_UPD == 0:
            self.t_net.copy_weights_from(self.q_net)


def _build_state(vs, prev_vs, use_momentum):
    if use_momentum:
        return np.concatenate([vs, vs - prev_vs])
    return vs.copy()


def run_loso_dqn(subjects, use_momentum=False, verbose=True):
    """
    LOSO with DQN.
    use_momentum=False → INN-1 (2-D continuous state)
    use_momentum=True  → INN-2 (4-D momentum state)

    Action = GEW emotion direction index (8, subject-independent).
    Clip resolved per-subject via best_clip().
    """
    state_dim = 4 if use_momentum else 2
    sids = sorted(subjects.keys())
    errors, state_paths, vec_paths = {}, {}, {}
    rng  = np.random.default_rng(1)

    label = "INN-2 Momentum DQN" if use_momentum else "INN-1 DQN"
    iterator = tqdm(sids, desc=label, disable=not verbose)

    for test_sid in iterator:
        train_sids = [s for s in sids if s != test_sid]
        agent = DQNAgent(state_dim=state_dim)

        for epoch in range(DQN_EPOCHS):
            for tr_sid in train_sids:
                clips   = subjects[tr_sid]
                vs      = _make_start_vec(rng)
                prev_vs = vs.copy()

                for step in range(N_ITERATIONS):
                    sv  = _build_state(vs, prev_vs, use_momentum)
                    # 1. DQN picks GEW emotion action index
                    a   = agent.act(sv, explore=True)
                    # 2. Resolve to best matching clip for that emotion direction
                    va  = clips[best_clip(clips, a, vs)]
                    vsp = transition(vs, va)
                    r   = reward_fn(vs, va)
                    svp = _build_state(vsp, vs, use_momentum)

                    agent.store(sv, a, r, svp)
                    agent.learn()

                    prev_vs = vs.copy()
                    vs      = vsp
                    if vec_to_state(vs) == TARGET_EMOTION: break

        # ── Greedy evaluation ────────────────────────────────
        clips   = subjects[test_sid]
        vs      = _make_start_vec()
        prev_vs = vs.copy()
        s       = vec_to_state(vs)
        errs, states, vecs = [], [s], [vs.copy()]

        for _ in range(N_ITERATIONS):
            sv      = _build_state(vs, prev_vs, use_momentum)
            a       = agent.act(sv, explore=False)
            va      = clips[best_clip(clips, a, vs)]
            prev_vs = vs.copy()
            vs      = transition(vs, va)
            s       = vec_to_state(vs)
            errs.append(angular_error(vs))
            states.append(s)
            vecs.append(vs.copy())

        errors[test_sid]      = np.array(errs)
        state_paths[test_sid] = states
        vec_paths[test_sid]   = vecs

    return errors, state_paths, vec_paths

# ══════════════════════════════════════════════════════════════════════════════
# 13.  INN-4: MULTI-TARGET EMOTION NAVIGATION
#
#      Trains a separate agent for each GEW emotion as target.
#      Action space = 8 GEW emotion directions (consistent with other methods).
# ══════════════════════════════════════════════════════════════════════════════
def run_multi_target(subjects, targets=None, epochs=MT_EPOCHS, verbose=True):
    """Train/evaluate a separate agent for each target emotion."""
    if targets is None: targets = EMOTION_NAMES
    sids    = sorted(subjects.keys())
    results = {}

    for tgt in targets:
        tgt_vec   = GEW_UNIT[tgt]
        final_err = []

        for test_sid in sids:
            train_sids = [s for s in sids if s != test_sid]
            agent = TabularQAgent()

            for epoch in range(epochs):
                for tr_sid in train_sids:
                    clips = subjects[tr_sid]
                    vs    = _make_start_vec()
                    s     = vec_to_state(vs)
                    for step in range(N_ITERATIONS):
                        # Action = GEW emotion index → resolve clip
                        a   = agent.act(s, explore=True)
                        va  = clips[best_clip(clips, a, vs)]
                        vsp = transition(vs, va)
                        sp  = vec_to_state(vsp)
                        r   = reward_fn(vs, va, target_vec=tgt_vec)
                        phi = _shaping_fixed(s, sp)
                        agent.update(s, a, r, phi, sp)
                        vs, s = vsp, sp
                        if s == tgt: break

            # Eval
            clips = subjects[test_sid]
            vs    = _make_start_vec()
            s     = vec_to_state(vs)
            for _ in range(N_ITERATIONS):
                a  = agent.act(s, explore=False)
                va = clips[best_clip(clips, a, vs)]
                vs = transition(vs, va)
                s  = vec_to_state(vs)
            final_err.append(angular_error(vs, target_vec=tgt_vec))

        results[tgt] = np.array(final_err)
        if verbose:
            nc = (results[tgt] < CONVERGE_THRESH).sum()
            print(f"    Target {tgt:>8}: mean={results[tgt].mean():.1f}°  conv={nc}/32")

    return results

# ══════════════════════════════════════════════════════════════════════════════
# 14.  TABLE I
# ══════════════════════════════════════════════════════════════════════════════
def compute_table1(errors):
    sids  = sorted(errors.keys())
    arr   = np.array([errors[s] for s in sids])
    final = arr[:, -1]
    converged = final < CONVERGE_THRESH

    S = arr[ converged]
    F = arr[~converged]
    T = arr

    col_groups = [slice(0, 2), slice(2, 3), slice(3, 4), slice(4, 5), slice(5, 6)]
    headers    = ["Clip 1-2", "3", "4", "5", "6"]

    rows = {}
    for lbl, sub in [("T", T), ("S", S), ("F", F)]:
        row = []
        for sl in col_groups:
            v = sub[:, sl].flatten()
            row.append(f"{v.mean():.1f}({v.std():.1f})" if len(v) else "N/A")
        rows[lbl] = row

    df = pd.DataFrame(rows, index=headers).T
    df.index.name = "Group"
    return df, np.where(converged)[0], np.where(~converged)[0]


def print_table1(errors, label=""):
    df, succ, fail = compute_table1(errors)
    sids  = sorted(errors.keys())
    arr   = np.array([errors[s] for s in sids])
    final = arr[:, -1]

    S_ids = [sids[i] for i in succ]
    F_ids = [sids[i] for i in fail]

    title = f"TABLE I — {label}" if label else "TABLE I"
    print("\n" + "═" * 65)
    print(f"  {title}  |  T=Total S=Success F=Fail")
    print("═" * 65)
    print(df.to_string())
    print(f"\n  Converged (<{CONVERGE_THRESH}°): {len(succ)}/32  "
          f"(clip-6 mean {final[succ].mean():.1f}°)" if len(succ) else
          f"\n  Converged: 0/32")
    print(f"  Failed: {len(fail)}/32  "
          f"(clip-6 mean {final[fail].mean():.1f}°)" if len(fail) else
          f"  Failed: 0/32")
    print(f"  Overall clip-6: {final.mean():.1f}° ± {final.std():.1f}°")
    return df, S_ids, F_ids

# ══════════════════════════════════════════════════════════════════════════════
# 15.  ANALYSIS HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def convergence_stats(errors, thresh=CONVERGE_THRESH):
    sids  = sorted(errors.keys())
    arr   = np.array([errors[s] for s in sids])
    final = arr[:, -1]
    mask  = final < thresh
    return {
        'n_conv':     int(mask.sum()),
        'pct_conv':   100.0 * mask.sum() / len(sids),
        'mean_final': float(final.mean()),
        'std_final':  float(final.std()),
        'mean_conv':  float(final[mask].mean())  if mask.any()    else np.nan,
        'mean_fail':  float(final[~mask].mean()) if (~mask).any() else np.nan,
        'curve':      arr.mean(0),
        'curve_std':  arr.std(0),
    }

def summary_df(stats_dict):
    rows = []
    for name, st in stats_dict.items():
        rows.append({
            'Method':      name,
            'Converged':   f"{st['n_conv']}/32",
            'Conv %':      f"{st['pct_conv']:.1f}",
            'Mean Final°': f"{st['mean_final']:.1f}",
            'Std°':        f"{st['std_final']:.1f}",
            'Mean Conv°':  f"{st['mean_conv']:.1f}" if not np.isnan(st['mean_conv']) else 'N/A',
            'Mean Fail°':  f"{st['mean_fail']:.1f}" if not np.isnan(st['mean_fail']) else 'N/A',
        })
    return pd.DataFrame(rows).set_index('Method')

# ══════════════════════════════════════════════════════════════════════════════
# 16.  GEW WHEEL HELPER
# ══════════════════════════════════════════════════════════════════════════════
def _draw_gew_wheel(ax, alpha_bg=0.12):
    ax.fill_betweenx([0, 1.2],  0,   1.2,  color="#FFFACD", alpha=alpha_bg)
    ax.fill_betweenx([0, 1.2], -1.2, 0,    color="#FFE4E1", alpha=alpha_bg)
    ax.fill_betweenx([-1.2, 0],-1.2, 0,    color="#E0E0FF", alpha=alpha_bg)
    ax.fill_betweenx([-1.2, 0], 0,   1.2,  color="#E8FFE8", alpha=alpha_bg)
    circle = plt.Circle((0, 0), 1.0, color="silver", fill=False, lw=1.0, ls="--")
    ax.add_patch(circle)
    ax.axhline(0, color="gray", lw=0.6, ls=":")
    ax.axvline(0, color="gray", lw=0.6, ls=":")
    for name, uvec in GEW_UNIT.items():
        x, y = uvec * 1.08
        c = EMOTION_COLORS[name]
        ax.scatter(*uvec, color=c, s=55, zorder=6, edgecolors="k", lw=0.4)
        ax.text(x, y, name[:4].capitalize(), fontsize=6, ha="center",
                va="center", color=c, fontweight="bold")
    ax.scatter(*TARGET_VEC, marker="*", s=180, color="gold",
               edgecolors="darkorange", lw=0.8, zorder=8)
    ax.set_xlim(-1.35, 1.35); ax.set_ylim(-1.35, 1.35)
    ax.set_aspect("equal"); ax.tick_params(labelsize=6)
    ax.set_xlabel("Valence →", fontsize=7)
    ax.set_ylabel("Arousal →", fontsize=7)

# ══════════════════════════════════════════════════════════════════════════════
# 17.  PLOTS
# ══════════════════════════════════════════════════════════════════════════════
PALETTE = {
    "Baseline (Tabular Q)":    ("#4878CF", "--"),
    "INN-3: Adaptive Shaping": ("#6ACC65", "-."),
    "INN-1: DQN (2-D state)":  ("#D65F5F", "-"),
    "INN-2: Momentum DQN":     ("#B47CC7", ":"),
}

def _method_style(name):
    return PALETTE.get(name, ("#888888", "-"))

# ── Fig 2: Angular error vs iterations (with/without shaping) ──────────────
def plot_fig2(err_with, err_without):
    iters   = np.arange(1, N_ITERATIONS + 1)
    mean_w  = np.mean([err_with[s]    for s in sorted(err_with)],   axis=0)
    mean_wo = np.mean([err_without[s] for s in sorted(err_without)], axis=0)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(iters, mean_wo, color="steelblue",  lw=2, ls="--", label="Without reward shaping")
    ax.plot(iters, mean_w,  color="darkorange", lw=2, ls="-.", label="With reward shaping")
    ax.set_xlabel("Number of iterations")
    ax.set_ylabel("Angular Error in Emotion Transition (degrees)")
    ax.set_title("Angular Error vs Iterations  (paper Fig. 2)")
    ax.set_xlim(0, N_ITERATIONS + 0.5); ax.set_ylim(0, 230)
    ax.legend()
    plt.tight_layout()
    plt.savefig(out("fig2_angular_error.png"), dpi=150); plt.close()
    print("  Saved: fig2_angular_error.png")

# ── Fig 3: Error bar plot ──────────────────────────────────────────────────
def plot_fig3(err_with):
    arr   = np.array([err_with[s] for s in sorted(err_with)])
    mean  = arr.mean(axis=0)
    se    = arr.std(axis=0) / np.sqrt(len(arr))
    iters = np.arange(1, N_ITERATIONS + 1)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(iters, mean, yerr=se, fmt="-o", color="black",
                capsize=5, lw=1.8, ms=5, elinewidth=1.5)
    ax.set_xlabel("# of iterations"); ax.set_ylabel("Angular Error (degrees)")
    ax.set_title("Error Bar Plot  (paper Fig. 3)")
    ax.set_xlim(0.5, 6.5); ax.set_xticks(iters)
    plt.tight_layout()
    plt.savefig(out("fig3_error_bar.png"), dpi=150); plt.close()
    print("  Saved: fig3_error_bar.png")

# ── Fig 4: Two-subject paths ───────────────────────────────────────────────
def plot_fig4(err_with, state_paths, S_ids, F_ids):
    if not S_ids or not F_ids:
        print("  Skipping fig4 (need ≥1 converged AND ≥1 failed subject)")
        return
    sids  = sorted(err_with.keys())
    arr   = np.array([err_with[s] for s in sids])
    final = arr[:, -1]
    good_sid = min(S_ids, key=lambda s: final[sids.index(s)])
    bad_sid  = max(F_ids, key=lambda s: final[sids.index(s)])

    fig, ax = plt.subplots(figsize=(9, 5))
    iters    = np.arange(N_ITERATIONS + 1)
    init_err = angular_error(_make_start_vec())

    for sid, color, label in [(good_sid, "tab:blue",   f"Subject {good_sid} (converged)"),
                               (bad_sid,  "tab:orange", f"Subject {bad_sid}  (oscillating)")]:
        path      = state_paths[sid]
        errs_full = [init_err] + list(err_with[sid])
        ax.plot(iters, errs_full, "o--", color=color, lw=1.8, label=label)
        for i, (err, em) in enumerate(zip(errs_full, path)):
            c = EMOTION_COLORS.get(em, "gray")
            ax.scatter(i, err, color=c, s=70, zorder=5)
            ax.annotate(em.capitalize(), (i, err),
                        textcoords="offset points", xytext=(5, 4),
                        fontsize=7, color=c, fontweight="bold")

    ax.set_xlabel("# of Iterations"); ax.set_ylabel("Angular Error (degrees)")
    ax.set_title("Angular Error and Emotional State — Two Subjects (paper Fig. 4)")
    ax.set_ylim(-5, 200); ax.set_xticks(iters); ax.legend()
    plt.tight_layout()
    plt.savefig(out("fig4_two_subjects.png"), dpi=150); plt.close()
    print("  Saved: fig4_two_subjects.png")

# ── Fig A: Innovation comparison curves ───────────────────────────────────
def plot_comparison_curves(results_dict, fname="inn_fig_A_curves.png"):
    iters = np.arange(1, N_ITERATIONS + 1)
    fig, ax = plt.subplots(figsize=(8, 5))

    for name, errors in results_dict.items():
        st = convergence_stats(errors)
        col, ls = _method_style(name)
        curve = st['curve']
        ci    = st['curve_std'] * 0.5
        ax.plot(iters, curve, lw=2.2, color=col, ls=ls,
                label=f"{name}  [conv {st['n_conv']}/32 | final {st['mean_final']:.1f}°]")
        ax.fill_between(iters, curve - ci, curve + ci, alpha=0.12, color=col)

    ax.axhline(CONVERGE_THRESH, color="gray", ls=":", lw=1.2,
               label=f"Convergence threshold ({CONVERGE_THRESH}°)")
    ax.set_xlabel("Iteration (music clip number)", fontsize=10)
    ax.set_ylabel("Mean Angular Error (degrees)", fontsize=10)
    ax.set_title("Innovation Comparison — Angular Error vs Iterations", fontsize=11)
    ax.legend(fontsize=8, loc="upper right")
    ax.set_xlim(0.7, N_ITERATIONS + 0.3); ax.set_ylim(0, 180); ax.set_xticks(iters)
    plt.tight_layout()
    plt.savefig(out(fname), dpi=150); plt.close()
    print(f"  Saved: {fname}")

# ── Fig B: Summary bar charts ──────────────────────────────────────────────
def plot_summary_bars(stats_dict, fname="inn_fig_B_bars.png"):
    methods     = list(stats_dict.keys())
    conv_pcts   = [stats_dict[m]['pct_conv']   for m in methods]
    mean_finals = [stats_dict[m]['mean_final'] for m in methods]
    colors      = [_method_style(m)[0] for m in methods]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
    b1 = ax1.bar(methods, conv_pcts, color=colors, edgecolor='k', lw=0.8, alpha=0.85)
    ax1.bar_label(b1, fmt='%.1f%%', padding=3, fontsize=9, fontweight='bold')
    ax1.set_ylabel("Subjects Converging (%)")
    ax1.set_title("Convergence Rate per Method")
    ax1.set_ylim(0, 105)
    ax1.tick_params(axis='x', rotation=15, labelsize=8)
    ax1.axhline(100 * 19 / 32, color='dimgray', ls='--', lw=1.2,
                label="Paper baseline (19/32 = 59.4%)")
    ax1.legend(fontsize=8)

    b2 = ax2.bar(methods, mean_finals, color=colors, edgecolor='k', lw=0.8, alpha=0.85)
    ax2.bar_label(b2, fmt='%.1f°', padding=3, fontsize=9, fontweight='bold')
    ax2.set_ylabel("Mean Final Angular Error (°)")
    ax2.set_title("Final Error at Iteration 6")
    ax2.tick_params(axis='x', rotation=15, labelsize=8)
    ax2.axhline(56.9, color='dimgray', ls='--', lw=1.2, label="Paper baseline (56.9°)")
    ax2.legend(fontsize=8)

    fig.suptitle("Innovation Method Comparison  (32 subjects, LOSO)",
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(out(fname), dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {fname}")

# ── Fig C: Box plot ────────────────────────────────────────────────────────
def plot_boxplot(results_dict, fname="inn_fig_C_boxplot.png"):
    fig, ax = plt.subplots(figsize=(9, 5))
    data    = [[errors[s][-1] for s in sorted(errors)] for errors in results_dict.values()]
    labels  = list(results_dict.keys())
    colors  = [_method_style(m)[0] for m in labels]

    bp = ax.boxplot(data, patch_artist=True, notch=False, widths=0.55,
                    medianprops=dict(color='black', lw=2.5))
    for patch, col in zip(bp['boxes'], colors):
        patch.set_facecolor(col); patch.set_alpha(0.7)
    for element in ['whiskers', 'caps', 'fliers']:
        for item in bp[element]:
            item.set_color('dimgray')

    ax.axhline(CONVERGE_THRESH, color='red', ls='--', lw=1.5,
               label=f"Convergence threshold ({CONVERGE_THRESH}°)")
    ax.set_xticklabels(labels, rotation=14, fontsize=8.5)
    ax.set_ylabel("Final Angular Error — Iteration 6 (degrees)")
    ax.set_title("Distribution of Final Errors per Method", fontsize=11)
    ax.legend(fontsize=9); ax.set_ylim(0, 200)
    plt.tight_layout()
    plt.savefig(out(fname), dpi=150); plt.close()
    print(f"  Saved: {fname}")

# ── Fig D: Multi-target radar ──────────────────────────────────────────────
def plot_multitarget_radar(multi_results, fname="inn_fig_D_radar.png"):
    targets = list(multi_results.keys())
    means   = [multi_results[t].mean() for t in targets]
    n_conv  = [(multi_results[t] < CONVERGE_THRESH).sum() for t in targets]
    N       = len(targets)
    angles  = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()

    fig = plt.figure(figsize=(13, 5.5))

    ax = plt.subplot(121, polar=True)
    vals = means + [means[0]]; angs = angles + [angles[0]]
    ax.plot(angs, vals, 'o-', lw=2.2, color='royalblue', ms=6)
    ax.fill(angs, vals, alpha=0.18, color='royalblue')
    ax.set_xticks(angles)
    ax.set_xticklabels([t.capitalize() for t in targets], fontsize=10, fontweight='bold')
    ax.set_ylim(0, 180)
    ax.set_title("Mean Final Angular Error\nby Target Emotion", fontsize=11, pad=20)
    for ang, val in zip(angles, means):
        ax.annotate(f"{val:.0f}°", xy=(ang, val), xytext=(ang, val + 18),
                    fontsize=8, ha='center', color='navy')

    ax2 = plt.subplot(122)
    bar_cols = [EMOTION_COLORS.get(t, "#888888") for t in targets]
    bars = ax2.bar(targets, [100 * c / 32 for c in n_conv],
                   color=bar_cols, edgecolor='k', lw=0.7, alpha=0.9)
    ax2.bar_label(bars, labels=[f"{c}/32" for c in n_conv],
                  padding=2, fontsize=9, fontweight='bold')
    ax2.set_ylabel("Subjects Converging (%)")
    ax2.set_title("Convergence Rate per Target Emotion")
    ax2.set_ylim(0, 105)
    ax2.tick_params(axis='x', rotation=15)

    fig.suptitle("INN-4: Multi-Target Emotion Navigation", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(out(fname), dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {fname}")

# ── Fig E: Adaptive shaping schedule ──────────────────────────────────────
def plot_shaping_lambda(fname="inn_fig_E_shaping.png"):
    iters        = np.arange(N_ITERATIONS)
    lam_fixed    = [LAMBDA_BASE] * N_ITERATIONS
    lam_adaptive = [LAMBDA_BASE * (1 + ADAPT_BETA * t / N_ITERATIONS) for t in iters]

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(iters + 1, lam_fixed,    'b--', lw=2, label="Fixed λ (baseline)")
    ax.plot(iters + 1, lam_adaptive, 'r-',  lw=2, label=f"Adaptive λ (INN-3, β={ADAPT_BETA})")
    ax.axhline(0, color='gray', lw=0.8, ls=':')
    ax.set_xlabel("Iteration"); ax.set_ylabel("Shaping penalty λ")
    ax.set_title("INN-3: Adaptive vs Fixed Reward Shaping Schedule", fontsize=10)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(out(fname), dpi=150); plt.close()
    print(f"  Saved: {fname}")

# ── Fig F: Per-subject improvement heat map ───────────────────────────────
def plot_improvement_heatmap(results_dict, fname="inn_fig_F_heatmap.png"):
    methods  = list(results_dict.keys())
    sids     = sorted(next(iter(results_dict.values())).keys())

    mat   = np.array([[results_dict[m][s][-1] for s in sids] for m in methods])
    delta = mat[0] - mat   # positive = innovation better than row 0

    fig, ax = plt.subplots(figsize=(14, 3.5 + 0.55 * len(methods)))
    im = ax.imshow(delta, cmap='RdYlGn', vmin=-60, vmax=60, aspect='auto')
    ax.set_yticks(range(len(methods)))
    ax.set_yticklabels(methods, fontsize=8)
    ax.set_xticks(range(len(sids)))
    ax.set_xticklabels([f"S{s:02d}" for s in sids], rotation=90, fontsize=7)
    ax.set_title("Δ Final Angular Error vs Baseline (green = innovation better)", fontsize=10)
    plt.colorbar(im, ax=ax, label="Improvement (°)")
    plt.tight_layout()
    plt.savefig(out(fname), dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {fname}")

# ── Fig G: State coverage comparison ──────────────────────────────────────
def plot_state_coverage(state_paths_dict, fname="inn_fig_G_coverage.png"):
    methods = list(state_paths_dict.keys())
    fig, axes = plt.subplots(1, len(methods), figsize=(4.5 * len(methods), 4))
    if len(methods) == 1: axes = [axes]

    for ax, (name, state_paths) in zip(axes, state_paths_dict.items()):
        counts = {}
        for sid, path in state_paths.items():
            for st in path:
                counts[st] = counts.get(st, 0) + 1
        total = sum(counts.values())
        states_sorted = sorted(counts, key=lambda x: counts[x], reverse=True)[:8]
        vals   = [100 * counts[s] / total for s in states_sorted]
        colors = [EMOTION_COLORS.get(s, "#888") for s in states_sorted]
        ax.barh(states_sorted, vals, color=colors, edgecolor='k', lw=0.6)
        ax.set_xlabel("Visit frequency (%)")
        ax.set_title(name, fontsize=9, fontweight='bold')
        ax.set_xlim(0, max(vals) * 1.3)
        for i, v in enumerate(vals):
            ax.text(v + 0.3, i, f"{v:.1f}%", va='center', fontsize=8)

    fig.suptitle("Emotional State Coverage During Evaluation", fontsize=11)
    plt.tight_layout()
    plt.savefig(out(fname), dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {fname}")

# ── Regret plot ────────────────────────────────────────────────────────────
def plot_regret(err_with, err_without):
    sids   = sorted(err_with.keys())
    reg_w  = np.cumsum(np.mean([err_with[s]    for s in sids], axis=0))
    reg_wo = np.cumsum(np.mean([err_without[s] for s in sids], axis=0))

    iters = np.arange(1, N_ITERATIONS + 1)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(iters, reg_wo, "b--", lw=2, label="Without reward shaping")
    ax.plot(iters, reg_w,  "r-.", lw=2, label="With reward shaping")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Cumulative Angular Error (Regret)")
    ax.set_title("Regret vs Iterations")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out("fig_regret.png"), dpi=150); plt.close()
    print("  Saved: fig_regret.png")

# ── Individual subject trajectory ─────────────────────────────────────────
def plot_subject_trajectory(sid, vec_paths, err_with, converged, save=True):
    vecs     = vec_paths[sid]
    init_err = angular_error(_make_start_vec())
    errs     = [init_err] + list(err_with[sid])

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    status = "CONVERGED" if converged else "FAILED"
    fig.suptitle(f"Subject {sid:02d} — {status}", fontsize=11, fontweight="bold",
                 color="green" if converged else "red")

    ax = axes[0]
    _draw_gew_wheel(ax)
    cmap = plt.get_cmap("plasma")
    for i in range(len(vecs) - 1):
        v0, v1 = np.array(vecs[i]), np.array(vecs[i + 1])
        frac   = i / max(len(vecs) - 2, 1)
        if not _is_neutral_vec(v0):
            ax.annotate("", xy=v1, xytext=v0,
                        arrowprops=dict(arrowstyle="-|>", color=cmap(frac),
                                        lw=1.5, mutation_scale=12))
        ax.scatter(*(v0 if not _is_neutral_vec(v0) else np.array([0.0, 0.0])),
                   color=cmap(frac), s=40, zorder=7, edgecolors="k", lw=0.3)

    ax.scatter(0, 0, s=80, color="lime", edgecolors="k", lw=0.6, zorder=9,
               marker="o", label="Start")
    if not _is_neutral_vec(vecs[-1]):
        ax.scatter(*vecs[-1], s=100, color="red", edgecolors="k", lw=0.6,
                   zorder=9, marker="X", label="End")
    ax.set_title("GEW Emotion Trajectory", fontsize=9)
    ax.legend(fontsize=7, loc="lower right")

    ax2 = axes[1]
    iters = np.arange(len(errs))
    col   = "green" if converged else "tomato"
    ax2.plot(iters, errs, "o-", color=col, lw=2, ms=6)
    ax2.axhline(CONVERGE_THRESH, color="gray", ls="--", lw=1,
                label=f"Threshold ({CONVERGE_THRESH}°)")
    ax2.fill_between(iters, 0, errs, alpha=0.12, color=col)
    ax2.set_xlabel("Iteration (0 = initial)", fontsize=9)
    ax2.set_ylabel("Angular Error (°)", fontsize=9)
    ax2.set_title("Angular Error vs Iteration", fontsize=9)
    ax2.set_xticks(iters); ax2.set_ylim(0, 200)
    ax2.legend(fontsize=7)

    states = [vec_to_state(v) for v in vecs]
    for i, (err, st) in enumerate(zip(errs, states)):
        c = EMOTION_COLORS.get(st, "gray")
        ax2.annotate(st[:4].capitalize(), (i, err),
                     textcoords="offset points", xytext=(4, 5),
                     fontsize=7, color=c, fontweight="bold")
    plt.tight_layout()
    if save:
        plt.savefig(out(f"trajectory_subject_{sid:02d}.png"), dpi=130)
        plt.close()
    return fig

# ── All-subjects grid ──────────────────────────────────────────────────────
def plot_all_trajectories_grid(err_with, vec_paths, S_ids, F_ids):
    sids  = sorted(err_with.keys())
    ncols = 8
    nrows = int(np.ceil(len(sids) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * 2.8, nrows * 2.8))
    fig.suptitle("GEW Emotion Trajectories — All Subjects\n"
                 "★ = target (Happy)  |  green border = converged  |  red = failed",
                 fontsize=11, y=1.01)

    cmap = plt.get_cmap("plasma")
    for idx, sid in enumerate(sids):
        row, col = divmod(idx, ncols)
        ax = axes[row][col] if nrows > 1 else axes[col]
        _draw_gew_wheel(ax, alpha_bg=0.08)
        vecs = vec_paths[sid]
        conv = sid in S_ids
        for i in range(len(vecs) - 1):
            v0, v1 = np.array(vecs[i]), np.array(vecs[i + 1])
            frac   = i / max(len(vecs) - 2, 1)
            if not _is_neutral_vec(v0):
                ax.annotate("", xy=v1, xytext=v0,
                            arrowprops=dict(arrowstyle="-|>", color=cmap(frac),
                                            lw=1.2, mutation_scale=10))
        ax.scatter(0, 0, s=30, color="lime", zorder=9, edgecolors="k", lw=0.3)
        if not _is_neutral_vec(vecs[-1]):
            ax.scatter(*vecs[-1], s=40, color="red", zorder=9, marker="X",
                       edgecolors="k", lw=0.3)
        border_col = "#00AA00" if conv else "#CC0000"
        for spine in ax.spines.values():
            spine.set_edgecolor(border_col); spine.set_linewidth(2.0)
        ax.set_title(f"S{sid:02d}", fontsize=7, pad=1,
                     color="green" if conv else "red", fontweight="bold")
        ax.set_xticklabels([]); ax.set_yticklabels([])
        ax.set_xlabel(""); ax.set_ylabel("")

    for idx in range(len(sids), nrows * ncols):
        row, col = divmod(idx, ncols)
        ax = axes[row][col] if nrows > 1 else axes[col]
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(out("fig_all_trajectories_grid.png"), dpi=130, bbox_inches="tight")
    plt.close()
    print("  Saved: fig_all_trajectories_grid.png")

# ── Action space diagram (from code 2) ────────────────────────────────────
def plot_action_space_diagram(fname="inn_fig_action_space.png"):
    """Show the 8 GEW emotion directions that form the fixed action space."""
    fig, ax = plt.subplots(figsize=(6, 6))
    _draw_gew_wheel(ax, alpha_bg=0.15)
    for i, (name, uvec) in enumerate(GEW_UNIT.items()):
        c = EMOTION_COLORS[name]
        ax.annotate("", xy=uvec * 0.95, xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color=c, lw=2.0,
                                    mutation_scale=14))
        ax.text(uvec[0] * 1.18, uvec[1] * 1.18,
                f"a{i}: {name}", fontsize=8, ha="center",
                va="center", color=c, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=c, alpha=0.8))
    ax.set_title("Action Space: 8 GEW Emotion Directions\n"
                 "(subject-independent; clip resolved per-subject via cosine similarity)",
                 fontsize=9)
    plt.tight_layout()
    plt.savefig(out(fname), dpi=150); plt.close()
    print(f"  Saved: {fname}")

# ══════════════════════════════════════════════════════════════════════════════
# 18.  REPORT
# ══════════════════════════════════════════════════════════════════════════════
def save_report(stats_dict, multi_results, table_df, err_with, state_paths, S_ids, F_ids):
    df    = summary_df(stats_dict)
    sids  = sorted(err_with.keys())
    arr   = np.array([err_with[s] for s in sids])
    final = arr[:, -1]

    lines = [
        "=" * 72,
        "  INNOVATION REPORT — RL Music Emotion Management",
        "  Reference: Dutta et al., IEEE EMBC 2020",
        "=" * 72,
        "",
        "DATA & ACTION SPACE",
        "─" * 72,
        "  Loading   : CSV only (z-score per-subject → tanh, code-1 pipeline)",
        "  Actions    : 8 GEW emotion directions (subject-independent)",
        "  Clip select: best_clip(clips, action_idx, vs) — cosine similarity",
        "               0.7 × target-emotion sim + 0.3 × current-state sim",
        "",
        "INNOVATIONS IMPLEMENTED",
        "─" * 72,
        "  INN-1  Continuous DQN (raw valence/arousal state, neural Q-function)",
        "  INN-2  Momentum DQN  (4-D state: v, a, Δv, Δa — trajectory-aware)",
        "  INN-3  Adaptive Reward Shaping (λ grows with iteration depth)",
        "  INN-4  Multi-Target Navigation (all 8 GEW emotions as targets)",
        "",
        "COMPARATIVE RESULTS  (LOSO, 32 subjects, 6 iterations)",
        "─" * 72,
        df.to_string(),
        "",
        "  Paper baseline (Table I, T-group): 19/32 converged, final ≈ 57°",
        "",
        "BASELINE TABLE I (with fixed shaping)",
        "─" * 72,
        table_df.to_string(),
        f"\n  Converged (<{CONVERGE_THRESH}°): {len(S_ids)}/32",
        f"  Failed: {len(F_ids)}/32",
        f"  Overall clip-6: {final.mean():.1f}° ± {final.std():.1f}°",
        "",
        "INN-4: MULTI-TARGET RESULTS",
        "─" * 72,
        f"  {'Target':<10} {'Mean Final°':>12}  {'Conv/32':>8}  {'Conv%':>7}",
    ]
    for tgt, errs in multi_results.items():
        nc = (errs < CONVERGE_THRESH).sum()
        lines.append(f"  {tgt:<10} {errs.mean():>11.1f}°  {nc:>8}  {100*nc/32:>6.1f}%")

    lines += [
        "",
        "PER-SUBJECT DETAIL (baseline with shaping)",
        "─" * 72,
        f"{'SID':>4}  {'Status':>10}  {'FinalErr':>9}  Emotion Path",
        "-" * 72,
    ]
    for i, sid in enumerate(sids):
        status = "CONVERGED" if sid in S_ids else "FAILED"
        path   = " → ".join(state_paths[sid])
        lines.append(f"{sid:>4}  {status:>10}  {final[i]:>8.1f}°  {path}")

    lines += [
        "",
        "KEY INSIGHTS",
        "─" * 72,
        "  • Fixed 8-action GEW space ensures consistent policy transfer across subjects.",
        "  • Continuous DQN removes the 9-bucket discretisation bottleneck.",
        "  • Momentum state (INN-2) exposes emotional velocity to the agent,",
        "    reducing oscillation in failing subjects.",
        "  • Adaptive shaping (INN-3) mimics curriculum learning: free early,",
        "    escalating pressure at decisive final clips.",
        "  • Multi-target analysis (INN-4) shows positive-valence targets",
        "    (happy, pride, interest) are structurally easier to reach.",
        "=" * 72,
    ]

    report = "\n".join(lines)
    with open(out("innovation_report.txt"), "w", encoding="utf-8") as f:
        f.write(report)
    print("\n" + report)
    print("  Saved: innovation_report.txt")

# ══════════════════════════════════════════════════════════════════════════════
# 19.  MAIN
# ══════════════════════════════════════════════════════════════════════════════
def main():
    print("\n" + "═" * 65)
    print("  FINAL INNOVATION MODULE — RL Music Emotion Management")
    print("  Dutta et al., IEEE EMBC 2020  — Extended")
    print("═" * 65)

    # ── Load data (CSV only) ─────────────────────────────────────────────────
    print("\n[0] Loading data …")
    subjects = load_deap()
    if subjects is None:
        print("  No DEAP CSV found → using synthetic data")
        subjects = make_synthetic()

    all_errors      = {}
    all_state_paths = {}
    all_vec_paths   = {}

    # ── Baseline ─────────────────────────────────────────────────────────────
    print("\n[1] Baseline: Tabular Q + fixed shaping …")
    e, sp, vp = run_loso_tabular(subjects, shaping_fn=_shaping_fixed)
    all_errors["Baseline (Tabular Q)"]      = e
    all_state_paths["Baseline (Tabular Q)"] = sp
    all_vec_paths["Baseline (Tabular Q)"]   = vp
    st = convergence_stats(e)
    print(f"    ✓  Converged: {st['n_conv']}/32   Final: {st['mean_final']:.1f}° ± {st['std_final']:.1f}°")

    # Paper figures based on baseline
    print("\n[2] Running baseline WITHOUT shaping (for Fig. 2 / regret) …")
    e_noshape, _, _ = run_loso_tabular(subjects, shaping_fn=lambda s, sp, **kw: 0.0)
    plot_fig2(e, e_noshape)
    plot_fig3(e)
    plot_regret(e, e_noshape)
    table_df, S_ids, F_ids = print_table1(e, label="Baseline Tabular Q")
    plot_fig4(e, all_state_paths["Baseline (Tabular Q)"], S_ids, F_ids)

    # Per-subject trajectories
    print("\n[3] Per-subject trajectory plots …")
    for sid in tqdm(sorted(e.keys()), desc="Trajectories"):
        plot_subject_trajectory(sid, vp, e, sid in S_ids)
    plot_all_trajectories_grid(e, vp, S_ids, F_ids)
    plot_action_space_diagram()

    # ── INN-3 ────────────────────────────────────────────────────────────────
    print("\n[4] INN-3: Tabular Q + Adaptive Reward Shaping …")
    e, sp, vp = run_loso_tabular(subjects, shaping_fn=_shaping_adaptive)
    all_errors["INN-3: Adaptive Shaping"]      = e
    all_state_paths["INN-3: Adaptive Shaping"] = sp
    all_vec_paths["INN-3: Adaptive Shaping"]   = vp
    st = convergence_stats(e)
    print(f"    ✓  Converged: {st['n_conv']}/32   Final: {st['mean_final']:.1f}° ± {st['std_final']:.1f}°")

    # ── INN-1 ────────────────────────────────────────────────────────────────
    print("\n[5] INN-1: Continuous DQN (2-D state) …")
    e, sp, vp = run_loso_dqn(subjects, use_momentum=False)
    all_errors["INN-1: DQN (2-D state)"]      = e
    all_state_paths["INN-1: DQN (2-D state)"] = sp
    all_vec_paths["INN-1: DQN (2-D state)"]   = vp
    st = convergence_stats(e)
    print(f"    ✓  Converged: {st['n_conv']}/32   Final: {st['mean_final']:.1f}° ± {st['std_final']:.1f}°")

    # ── INN-2 ────────────────────────────────────────────────────────────────
    print("\n[6] INN-2: Momentum DQN (4-D state) …")
    e, sp, vp = run_loso_dqn(subjects, use_momentum=True)
    all_errors["INN-2: Momentum DQN"]      = e
    all_state_paths["INN-2: Momentum DQN"] = sp
    all_vec_paths["INN-2: Momentum DQN"]   = vp
    st = convergence_stats(e)
    print(f"    ✓  Converged: {st['n_conv']}/32   Final: {st['mean_final']:.1f}° ± {st['std_final']:.1f}°")

    # ── INN-4 ────────────────────────────────────────────────────────────────
    print("\n[7] INN-4: Multi-Target Navigation (all 8 GEW emotions) …")
    multi_results = run_multi_target(subjects, verbose=True)

    # ── Stats & innovation plots ──────────────────────────────────────────────
    print("\n[8] Aggregating statistics …")
    all_stats = {name: convergence_stats(err) for name, err in all_errors.items()}
    print("\n" + summary_df(all_stats).to_string())

    print("\n[9] Innovation figures …")
    plot_comparison_curves(all_errors)
    plot_summary_bars(all_stats)
    plot_boxplot(all_errors)
    plot_multitarget_radar(multi_results)
    plot_shaping_lambda()
    plot_improvement_heatmap(all_errors)
    subset_sp = {k: v for k, v in all_state_paths.items()
                 if k in ["Baseline (Tabular Q)", "INN-2: Momentum DQN"]}
    plot_state_coverage(subset_sp)

    # ── Report ────────────────────────────────────────────────────────────────
    print("\n[10] Writing report …")
    baseline_err = all_errors["Baseline (Tabular Q)"]
    baseline_sp  = all_state_paths["Baseline (Tabular Q)"]
    save_report(all_stats, multi_results, table_df, baseline_err,
                baseline_sp, S_ids, F_ids)

    print("\n" + "═" * 65)
    print("  ALL OUTPUTS saved to ./outputs_innovation/")
    print("─" * 65)
    for f in [
        "fig2_angular_error.png          — Paper Fig. 2 (with/without shaping)",
        "fig3_error_bar.png              — Paper Fig. 3",
        "fig4_two_subjects.png           — Paper Fig. 4",
        "fig_regret.png                  — Regret comparison",
        "fig_all_trajectories_grid.png   — All-subject grid",
        "trajectory_subject_NN.png       — 32 individual trajectory plots",
        "inn_fig_action_space.png        — Action space diagram (8 GEW directions)",
        "inn_fig_A_curves.png            — Innovation: error vs iterations",
        "inn_fig_B_bars.png              — Innovation: convergence bar chart",
        "inn_fig_C_boxplot.png           — Innovation: box plots",
        "inn_fig_D_radar.png             — INN-4 multi-target radar",
        "inn_fig_E_shaping.png           — INN-3 adaptive λ schedule",
        "inn_fig_F_heatmap.png           — Per-subject improvement heatmap",
        "inn_fig_G_coverage.png          — State coverage comparison",
        "innovation_report.txt           — Full written report",
    ]:
        print(f"  {f}")
    print("═" * 65)


if __name__ == "__main__":
    main()


═════════════════════════════════════════════════════════════════
  FINAL INNOVATION MODULE — RL Music Emotion Management
  Dutta et al., IEEE EMBC 2020  — Extended
═════════════════════════════════════════════════════════════════

[0] Loading data …
  Loaded DEAP (CSV): 32 subjects, 40 clips each.

[1] Baseline: Tabular Q + fixed shaping …


Tabular LOSO: 100%|██████████| 32/32 [05:31<00:00, 10.35s/it]


    ✓  Converged: 22/32   Final: 16.2° ± 28.5°

[2] Running baseline WITHOUT shaping (for Fig. 2 / regret) …


Tabular LOSO: 100%|██████████| 32/32 [16:08<00:00, 30.27s/it]


  Saved: fig2_angular_error.png
  Saved: fig3_error_bar.png
  Saved: fig_regret.png

═════════════════════════════════════════════════════════════════
  TABLE I — Baseline Tabular Q  |  T=Total S=Success F=Fail
═════════════════════════════════════════════════════════════════
         Clip 1-2           3           4           5           6
Group                                                            
T      24.4(24.7)  26.8(24.9)  14.5(27.3)  31.7(36.4)  16.2(28.5)
S      23.5(22.8)  18.1(19.5)    3.1(8.6)  19.3(20.8)    1.6(2.2)
F      26.2(28.3)  45.9(24.8)  39.8(36.1)  59.1(46.9)  48.1(33.2)

  Converged (<10.0°): 22/32  (clip-6 mean 1.6°)
  Failed: 10/32  (clip-6 mean 48.1°)
  Overall clip-6: 16.2° ± 28.5°
  Saved: fig4_two_subjects.png

[3] Per-subject trajectory plots …


...

    ✓  Converged: 17/32   Final: 33.2° ± 49.5°

[5] INN-1: Continuous DQN (2-D state) …


INN-1 DQN: 100%|██████████| 32/32 [01:18<00:00,  2.44s/it]


    ✓  Converged: 30/32   Final: 5.0° ± 17.3°

[6] INN-2: Momentum DQN (4-D state) …


INN-2 Momentum DQN: 100%|██████████| 32/32 [01:32<00:00,  2.90s/it]


    ✓  Converged: 17/32   Final: 41.9° ± 44.3°

[7] INN-4: Multi-Target Navigation (all 8 GEW emotions) …
    Target  sadness: mean=108.2°  conv=7/32
    Target    pride: mean=49.9°  conv=0/32
    Target interest: mean=50.8°  conv=2/32
    Target    happy: mean=31.7°  conv=20/32
    Target    guilt: mean=130.6°  conv=0/32
    Target  disgust: mean=133.8°  conv=1/32
    Target    anger: mean=106.3°  conv=0/32
    Target   relief: mean=92.3°  conv=0/32

[8] Aggregating statistics …

                        Converged Conv % Mean Final°  Std° Mean Conv° Mean Fail°
Method                                                                          
Baseline (Tabular Q)        22/32   68.8        16.2  28.5        1.6       48.1
INN-3: Adaptive Shaping     17/32   53.1        33.2  49.5        1.7       68.8
INN-1: DQN (2-D state)      30/32   93.8         5.0  17.3        0.8       69.2
INN-2: Momentum DQN         17/32   53.1        41.9  44.3        0.8       88.5

[9] Innovation figures …
  